## Лабораторный стенд: моделирование цифровой линии связи

Параметризуемая модель линии связи «передатчик → канал → приёмник» с оценкой
помехоустойчивости. Один и тот же стенд годится для разных схем модуляции и
разных условий канала: меняются только параметры.

Задача, которую решает стенд: быстро получить обоснованные цифры вместо
интуиции. Сколько нужно отношения сигнал/шум для заданной вероятности ошибки?
Какая получится полоса? Хватит ли разрядности при переносе в цифровую
технику? Что именно сломается, если уйдёт частота или «поплывёт» такт?

### Что даёт стенд

- расчёт производных параметров линии (полоса, частоты, запас по дискретизации);
- автоматическая проверка согласованности параметров;
- измерение BER и сравнение с теоретической кривой;
- диагностика по созвездию, глазковой диаграмме, спектру;
- оценка требований к разрядностям для аппаратной реализации;
- техническая сводка и сохранение результатов сессии.

### Как пользоваться

1. Раздел 1 — задать параметры (или выбрать готовый набор).
2. Раздел 2 — посмотреть, что из них получается, и проверить на противоречия.
3. Разделы 3–6 — прогнать модель и разобрать метрики.
4. Разделы 7–9 — выгрузить данные для реализации и сохранить отчёт.

Всё считается автоматически; править нужно только раздел 1.

### 1. Параметры

Единственное место, которое вы правите. Рядом в комментариях — типичные
значения и на что параметр влияет.

**Готовые наборы** (пресеты) — стартовые точки под типовые сценарии:

| Набор | Что моделирует | Ожидаемый BER |
|---|---|---|
| `qpsk_1m` | базовая линия, хорошие условия | совпадает с теорией |
| `qpsk_10m` | высокая скорость, расстройка частоты, ошибка такта | совпадает с теорией |
| `bpsk_reserve` | помехоустойчивый режим при слабом сигнале | совпадает с теорией |
| `uqpsk_6db` | два потока с разным приоритетом в одной полосе | совпадает с теорией |
| `oqpsk` | режим с пониженным пик-фактором | совпадает с теорией |
| `harsh` | жёсткие условия: слабый сигнал, расстройка, АЦП 6 бит | совпадает (всё компенсируется) |
| `multipath` | многолучёвость без эквалайзера | **плохой — так и должно быть** |
| `custom` | ничего не применять, только свои значения | — |

Наборы описаны в `dsp/config.py` (`PRESETS`), поэтому одинаково работают и в
ноутбуке, и в скрипте, и в автотестах. Свою задачу удобно оформить как новый
набор: тогда её можно переиспользовать и проверять.

In [1]:
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / "dsp" / "__init__.py").exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd

from dsp.config import ExperimentConfig, apply_preset, list_presets
from dsp import chain, report
from dsp.rx import metrics, sync, demod
from dsp.signals import sources, modulation, pulse, channel
from dsp.core import spectra, filters
from dsp.fpga import fixed as fx, export
from dsp.viz import plots
from dsp.units import esn0_to_ebn0_db

# =============================================================================
# ПРЕСЕТ
# =============================================================================
PRESET = "qpsk_1m"

# =============================================================================
# 1.1 СИГНАЛ — что передаём
# =============================================================================
MODULATION = "QPSK"        # "BPSK" | "QPSK" | "OQPSK" | "UQPSK"
BIT_RATE = 1.0e6           # бит/с — требуемая скорость
NUM_BITS = 200_000         # бит за прогон. Для BER 1e-5 нужно >= 3e5
SPS = 8                    # отсчётов на символ: 4..16 (8 — разумный компромисс)
ROLLOFF = 0.35             # α формирующего фильтра: 0.2..0.35 (меньше — уже полоса)
SPAN = 8                   # длина RRC в символах: 6..12 (больше — лучше фильтр)
SEED = 42                  # зерно генератора битов (воспроизводимость)
IQ_POWER_RATIO_DB = 0.0    # только UQPSK: насколько I мощнее Q (дБ). 0 = обычный QPSK

# =============================================================================
# 1.2 КАНАЛ — что портит сигнал
# =============================================================================
ES_N0_DB = 10.0            # Es/N0, дБ. None = без шума (проверка тракта)
CFO_HZ = 0.0               # расстройка несущей, Гц (доплер + ppm гетеродина)
PHASE_DEG = 0.0            # постоянный набег фазы, градусы
TIMING_OFFSET_SYMBOLS = 0.0  # ошибка момента стробирования, в символах (0..1)
PHASE_NOISE_DEG = 0.0      # фазовый шум (шум синтезатора), градусы СКЗ
MULTIPATH = ()             # лучи: ((задержка_симв, усиление_дБ, фаза_град), ...)
IQ_GAIN_DB = 0.0           # разбаланс амплитуд I/Q, дБ
IQ_PHASE_DEG = 0.0         # квадратурная ошибка, градусы
ADC_BITS = 0               # 0 = идеальный АЦП, иначе разрядность (например 12)
ADC_FULL_SCALE = 1.5       # полная шкала АЦП (в единицах сигнала)
CHANNEL_SEED = 7           # зерно шума (меняйте для независимых точек BER)

# =============================================================================
# 1.3 ПРИЁМНИК — какие блоки включены
# =============================================================================
MATCHED_FILTER = True      # согласованный фильтр RRC (нужен всегда)
COARSE_CFO = True          # грубая оценка частоты (метод M-й степени)
TIMING_RECOVERY = True     # тактовая синхронизация (Оэрдер-Мейер)
FINE_CARRIER = True        # петля Костаса (фаза)
RESOLVE_AMBIGUITY = True   # снятие неоднозначности 90° (по эталону, как маркер)
LOOP_BW_CARRIER = 0.01     # полоса петли Костаса (0.002..0.02)
LOOP_DAMPING = 0.707       # затухание петли

# =============================================================================
# 1.4 ПЛИС — форматы чисел для переноса в VHDL
# =============================================================================
ADC_WL, ADC_FL = 14, 10    # АЦП: 14 бит, 10 дробных -> шкала ±8
DATA_WL, DATA_FL = 16, 13  # тракт данных: ±4, шаг 1/8192
COEFF_WL, COEFF_FL = 16, 14  # коэффициенты фильтра
ROUNDING = "round"         # "round" | "trunc"
SATURATE = True            # True = насыщение (правильно), False = перенос (опасно)

# =============================================================================
# 1.5 ЧТО СДЕЛАТЬ
# =============================================================================
RUN_SWEEP = True           # считать кривую BER
SWEEP_ES_N0 = [4, 6, 8, 10, 12]   # точки свипа по Es/N0, дБ
SWEEP_BITS = 60_000        # бит на точку свипа (больше — точнее, но дольше)
CHECK_FIXED_POINT = True   # сравнить float и fixed-point
EXPORT_FPGA = True         # выгрузить .coe и тест-векторы
SAVE_SESSION = True        # сохранить конфиг, протокол и графики в out/

# =============================================================================
# ПРЕСЕТЫ: берём из библиотеки (dsp/config.py), чтобы они были одинаковыми
# в ноутбуке, в скрипте и в тестах
# =============================================================================
print("Доступные пресеты:")
print(list_presets())
print()

if PRESET != "custom":
    # Применяем пресет к черновому конфигу и читаем значения обратно в
    # переменные выше. Так вы правите именно переменные — они остаются
    # главными, а пресет служит только стартовой точкой.
    _pre = apply_preset(ExperimentConfig(), PRESET)
    MODULATION = _pre.signal.modulation
    BIT_RATE = _pre.signal.bit_rate
    NUM_BITS = _pre.signal.num_bits
    SPS = _pre.signal.sps
    ROLLOFF = _pre.signal.rolloff
    SPAN = _pre.signal.span
    SEED = _pre.signal.seed
    IQ_POWER_RATIO_DB = _pre.signal.iq_power_ratio_db
    ES_N0_DB = _pre.channel.es_n0_db
    CFO_HZ = _pre.channel.cfo_hz
    PHASE_DEG = _pre.channel.phase_deg
    TIMING_OFFSET_SYMBOLS = _pre.channel.timing_offset_symbols
    PHASE_NOISE_DEG = _pre.channel.phase_noise_deg
    MULTIPATH = _pre.channel.multipath
    IQ_GAIN_DB = _pre.channel.iq_gain_db
    IQ_PHASE_DEG = _pre.channel.iq_phase_deg
    ADC_BITS = _pre.channel.adc_bits
    ADC_FULL_SCALE = _pre.channel.adc_full_scale
    CHANNEL_SEED = _pre.channel.seed

# =============================================================================
# СБОРКА КОНФИГА (дальше не трогать — здесь только раскладка по структурам)
# =============================================================================
cfg = ExperimentConfig()
cfg.name = f"{PRESET}: {MODULATION} {BIT_RATE/1e6:.1f} Мбит/с"

cfg.signal.modulation = MODULATION
cfg.signal.bit_rate = BIT_RATE
cfg.signal.num_bits = NUM_BITS
cfg.signal.sps = SPS
cfg.signal.rolloff = ROLLOFF
cfg.signal.span = SPAN
cfg.signal.seed = SEED
cfg.signal.iq_power_ratio_db = IQ_POWER_RATIO_DB

cfg.channel.es_n0_db = ES_N0_DB
cfg.channel.cfo_hz = CFO_HZ
cfg.channel.phase_deg = PHASE_DEG
cfg.channel.timing_offset_symbols = TIMING_OFFSET_SYMBOLS
cfg.channel.phase_noise_deg = PHASE_NOISE_DEG
cfg.channel.multipath = MULTIPATH
cfg.channel.iq_gain_db = IQ_GAIN_DB
cfg.channel.iq_phase_deg = IQ_PHASE_DEG
cfg.channel.adc_bits = ADC_BITS
cfg.channel.adc_full_scale = ADC_FULL_SCALE
cfg.channel.seed = CHANNEL_SEED

cfg.receiver.matched_filter = MATCHED_FILTER
cfg.receiver.coarse_cfo = COARSE_CFO
cfg.receiver.timing_recovery = TIMING_RECOVERY
cfg.receiver.fine_carrier = FINE_CARRIER
cfg.receiver.resolve_ambiguity = RESOLVE_AMBIGUITY
cfg.receiver.loop_bw_carrier = LOOP_BW_CARRIER
cfg.receiver.loop_damping = LOOP_DAMPING

cfg.fpga.adc_wl, cfg.fpga.adc_fl = ADC_WL, ADC_FL
cfg.fpga.data_wl, cfg.fpga.data_fl = DATA_WL, DATA_FL
cfg.fpga.coeff_wl, cfg.fpga.coeff_fl = COEFF_WL, COEFF_FL
cfg.fpga.rounding = ROUNDING
cfg.fpga.saturate = SATURATE

cfg.validate()
print(f"Пресет: {PRESET}")
cfg.show()

Доступные пресеты:
  qpsk_1m        базовый QPSK 1 Мбит/с, хороший канал — проверка тракта
  qpsk_10m       QPSK 10 Мбит/с с расстройкой частоты и ошибкой такта
  bpsk_reserve   резервный BPSK 1 Мбит/с при слабом сигнале
  uqpsk_6db      UQPSK с перекосом 6 дБ (данные + служебный канал)
  oqpsk          OQPSK: меньше пик-фактор (линейность усилителя)
  harsh          жёсткие условия: низкое ОСШ, доплер, ошибка такта, АЦП 6 бит
  multipath      многолучёвость без эквалайзера (так BER и должен упасть)

Пресет: qpsk_1m
ЭКСПЕРИМЕНТ: qpsk_1m: QPSK 1.0 Мбит/с
Схема модуляции ....... QPSK (2 бит/символ)
Скорость .............. 1.000 Мбит/с = 0.500 Мбод
Дискретизация ......... 4.000 МГц (sps=8, 800000 отсчётов)
Фильтр RRC ............ α=0.35, ±4 символов, 65 отводов
Канал ................. Es/N0=12.0 дБ, CFO=0.0 Гц, фаза=0.0°, тайминг=0.0 симв.
Приёмник .............. МФ=True, CFO=True, Костас=True, тайминг=True
ПЛИС .................. данные 16 бит (fl=13), коэффициенты 16 бит


### 2. Производные параметры и проверка

Прежде чем считать вероятность ошибки, полезно посмотреть на **производные
величины**: какая получилась полоса, хватает ли частоты дискретизации,
влезает ли сигнал в выбранный формат чисел, не будет ли ограничения в АЦП.

Ниже — таблица и **список предупреждений**. Пустой список означает, что
параметры согласованы между собой.

In [2]:
params_df = pd.DataFrame(report.derived_table(cfg))
params_df.index = np.arange(1, len(params_df) + 1)
print("ПРОИЗВОДНЫЕ ВЕЛИЧИНЫ")
print(params_df.to_string(max_colwidth=60))

warns = report.check_config(cfg)
print()
print("ПРОВЕРКА ПАРАМЕТРОВ")
if warns:
    for w in warns:
        print(f"  ! {w}")
else:
    print("  ок — параметры согласованы, противоречий не найдено")

ПРОИЗВОДНЫЕ ВЕЛИЧИНЫ
                      Параметр      Значение                        Пояснение
1                    Модуляция          QPSK                     2 бит/символ
2             Битовая скорость  1.000 Мбит/с               требуемая скорость
3     Символьная скорость (Rs)    0.500 Мбод            бит/с ÷ бит на символ
4     Отсчётов на символ (sps)             8       передискретизация в тракте
5   Частота дискретизации (fs)     4.000 МГц        Rs × sps — такт обработки
6            Занимаемая полоса     0.675 МГц                 Rs×(1+α), α=0.35
7             Полоса Найквиста    ±2.000 МГц               fs/2 — выше нельзя
8       Запас по дискретизации         5.93×        fs / полоса (нужно ≥ 1.2)
9                Отводов в RRC            65               8 символов × 8 + 1
10                Задержка RRC    4.00 симв.          компенсируется в модели
11         Длительность записи     200.00 мс                  800000 отсчётов
12                Передано бит       200 00

### 3. Прогон и метрики

Один прогон всего тракта: биты → модулятор → формирующий фильтр → канал →
приёмник (фильтр, синхронизация) → решения → вероятность ошибки.

**Как читать результат:**

- `BER модели` должен совпадать с `BER теории` в пределах 1.5–2 раз.
  Заметно хуже — ищите причину в синхронизации или в разрядностях.
- `EVM` ≈ 100 %/√(Es/N0): при 10 дБ это около 32 %.
- `Ошибка оценки CFO` — точность оценки частоты, ожидаются единицы герц.
- `Оценка сдвига такта` — найденный момент стробирования, в отсчётах.

In [3]:
import time

t0 = time.perf_counter()
res = chain.run(cfg)
elapsed = time.perf_counter() - t0

theory = float(metrics.ber_theory_from_esn0(
    cfg.signal.modulation, cfg.channel.es_n0_db, cfg.signal.iq_power_ratio_db))
ratio = res.ber / theory if theory > 0 else float("inf")

d = res.rx.diagnostics
rows = [
    ("Схема модуляции", f"{cfg.signal.modulation}, {cfg.signal.bits_per_symbol} бит/символ"),
    ("Es/N0 (задано)", f"{cfg.channel.es_n0_db:.2f} дБ"),
    ("Eb/N0 (пересчёт)", f"{res.eb_n0_db:.2f} дБ"),
    ("Ошибок / бит", f"{res.errors} из {res.n_bits:,}".replace(",", " ")),
    ("BER модели", f"{res.ber:.3e}"),
    ("BER теории (АБГШ)", f"{theory:.3e}"),
    ("Отношение модель/теория", f"{ratio:.2f}x"),
    ("EVM", f"{res.evm_percent:.1f} %"),
    ("Оценка Es/N0 по символам", f"{res.es_n0_est_db:.2f} дБ"),
    ("Оценка CFO", f"{d.get('CFO оценено, Гц', float('nan')):,.0f} Гц"),
    ("Ошибка оценки CFO", f"{d.get('CFO ошибка, Гц', float('nan')):,.1f} Гц"),
    ("Оценка сдвига такта", f"{d.get('сдвиг такта, отсч.', 0):.3f} отсч."),
    ("Время прогона", f"{elapsed:.2f} с"),
]
print("=" * 62)
print(f"РЕЗУЛЬТАТ ПРОГОНА: {cfg.name}")
print("=" * 62)
for k, v in rows:
    print(f"{k:28s} {v:>28s}")
print("=" * 62)

# вердикт
print()
if res.ber <= 2.0 * theory:
    print(f"ВЕРДИКТ: тракт работает. BER в норме ({ratio:.2f}x от теории).")
elif res.ber <= 5.0 * theory:
    print(f"ВЕРДИКТ: есть потери {ratio:.1f}x относительно теории. Типичные причины:")
    print("  * слишком узкая/широкая полоса петли Костаса (LOOP_BW_CARRIER)")
    print("  * ошибка такта велика для выбранного sps")
    print("  * мало разрядов в формате данных / АЦП")
else:
    print(f"ВЕРДИКТ: тракт не работает ({ratio:.1f}x от теории). Проверьте по порядку:")
    print("  1) выключите все искажения (CFO=0, тайминг=0, шум=None) — ошибок быть не должно")
    print("  2) включите только шум — BER должен совпасть с теорией")
    print("  3) включайте искажения по одному и смотрите, после какого ломается")

РЕЗУЛЬТАТ ПРОГОНА: qpsk_1m: QPSK 1.0 Мбит/с
Схема модуляции                        QPSK, 2 бит/символ
Es/N0 (задано)                                   12.00 дБ
Eb/N0 (пересчёт)                                  8.99 дБ
Ошибок / бит                                 5 из 199 998
BER модели                                      2.500e-05
BER теории (АБГШ)                               3.430e-05
Отношение модель/теория                             0.73x
EVM                                                25.4 %
Оценка Es/N0 по символам                         11.96 дБ
Оценка CFO                                           0 Гц
Ошибка оценки CFO                                  0.0 Гц
Оценка сдвига такта                           7.996 отсч.
Время прогона                                      1.11 с

ВЕРДИКТ: тракт работает. BER в норме (0.73x от теории).


### 4. Диагностика: графики по этапам

Сначала общая картина — сигнал во времени и его спектр. Проверяем, что
занимаемая полоса соответствует расчётной, а вне полосы спектр спадает.

In [4]:
fs = cfg.signal.sample_rate
wave_tx = res.tx.waveform
wave_rx = res.channel.waveform

plots.show(plots.fig_time(wave_tx[:40 * cfg.signal.sps], fs,
                          "Переданный сигнал (I и Q, 40 символов)"))

plots.show(plots.fig_spectrum_compare(
    [("передано", wave_tx[:400 * cfg.signal.sps]),
     ("принято (с шумом)", wave_rx[:400 * cfg.signal.sps])],
    fs, "Спектр: переданный и принятый сигнал"))

Главный инструмент отладки — **созвездие**. Сравниваем «до» и «после»
синхронизации: до — точки размазаны и повёрнуты, после — должны собраться в
кучки вокруг идеальных позиций.

In [5]:
sym_before = res.rx.symbols_unsync
sym_after = res.rx.symbols

plots.show(plots.fig_constellation(
    sym_before, cfg.signal.modulation,
    f"Созвездие ДО синхронизации (EVM {demod.estimate_evm_percent(sym_before, cfg.signal.modulation, cfg.signal.iq_power_ratio_db):.0f}%)",
    iq_power_ratio_db=cfg.signal.iq_power_ratio_db))

plots.show(plots.fig_constellation(
    sym_after, cfg.signal.modulation,
    f"Созвездие ПОСЛЕ синхронизации (EVM {res.evm_percent:.0f}%)",
    iq_power_ratio_db=cfg.signal.iq_power_ratio_db))

plots.show(plots.fig_constellation(
    sym_after, cfg.signal.modulation, "Плотность точек (видно, где кучки)",
    density=True, iq_power_ratio_db=cfg.signal.iq_power_ratio_db))

**Глазковая диаграмма** и **АЧХ фильтра** — проверка формирования сигнала.
Глаз должен быть открыт, групповая задержка — ровная линия (признак линейной
фазы у симметричного фильтра).

In [6]:
mf_out = res.rx.stages.get("после_МФ")
if mf_out is not None:
    plots.show(plots.fig_eye(mf_out, cfg.signal.sps, 2,
                             "Глазковая диаграмма после согласованного фильтра"))

plots.show(plots.fig_filter_response(
    res.tx.taps, fs, f"АЧХ и групповая задержка RRC (α={cfg.signal.rolloff})",
    symbol_rate=cfg.signal.symbol_rate))

Если включена подстройка фазы — смотрим, как петля втягивается. По этому
графику видно, хватает ли её полосы: слишком узкая не успевает, слишком
широкая добавляет шума.

In [7]:
phase_hist = res.rx.stages.get("история_фазы")
if phase_hist is not None:
    plots.show(plots.fig_phase_track(
        {"история_фазы": res.rx.stages["история_фазы"],
         "история_частоты": res.rx.stages["история_частоты"]},
        "Работа петли Костаса"))
else:
    print("Петля Костаса выключена (FINE_CARRIER = False) — график не строится.")

### 5. Помехоустойчивость: кривая BER

Свип по отношению сигнал/шум: для каждой точки — отдельный прогон со своим
зерном шума. Результат: таблица с доверительными интервалами и график вместе
с теоретической кривой. Именно этот график обычно и становится главным
результатом работы.

In [8]:
if RUN_SWEEP:
    sweep_cfg = cfg.copy()
    sweep_cfg.signal.num_bits = SWEEP_BITS
    sweep_rows = chain.run_sweep(sweep_cfg, SWEEP_ES_N0, verbose=True)

    print()
    print(f"{'Es/N0':>7s} {'Eb/N0':>7s} {'ошибок':>8s} {'BER':>11s} "
          f"{'95% интервал':>24s} {'EVM%':>7s}")
    for r in sweep_rows:
        ci = f"[{r['ber_lo']:.2e}; {r['ber_hi']:.2e}]"
        print(f"{r['es_n0_db']:7.1f} {r['eb_n0_db']:7.2f} {r['errors']:8d} "
              f"{r['ber']:11.3e} {ci:>24s} {r['evm_percent']:7.1f}")

    plots.show(plots.fig_ber(sweep_rows, cfg.signal.modulation,
                             f"Кривая BER: {cfg.signal.modulation}",
                             iq_power_ratio_db=cfg.signal.iq_power_ratio_db))
else:
    sweep_rows = []
    print("Свип выключен (RUN_SWEEP = False)")

  Es/N0=  4.00 дБ  Eb/N0=  0.99 дБ  BER=4.330e-01  ошибок 25981/59999  EVM=59.7%  EsN0_оц=3.98 дБ


  Es/N0=  6.00 дБ  Eb/N0=  2.99 дБ  BER=2.570e-02  ошибок 1542/60000  EVM=49.0%  EsN0_оц=6.01 дБ


  Es/N0=  8.00 дБ  Eb/N0=  4.99 дБ  BER=6.800e-03  ошибок 408/59998  EVM=39.9%  EsN0_оц=8.01 дБ


  Es/N0= 10.00 дБ  Eb/N0=  6.99 дБ  BER=1.050e-03  ошибок 63/60000  EVM=31.9%  EsN0_оц=10.03 дБ


  Es/N0= 12.00 дБ  Eb/N0=  8.99 дБ  BER=3.333e-05  ошибок 2/60000  EVM=25.4%  EsN0_оц=12.00 дБ

  Es/N0   Eb/N0   ошибок         BER             95% интервал    EVM%
    4.0    0.99    25981   4.330e-01     [4.29e-01; 4.37e-01]    59.7
    6.0    2.99     1542   2.570e-02     [2.45e-02; 2.70e-02]    49.0
    8.0    4.99      408   6.800e-03     [6.17e-03; 7.49e-03]    39.9
   10.0    6.99       63   1.050e-03     [8.21e-04; 1.34e-03]    31.9
   12.0    8.99        2   3.333e-05     [9.14e-06; 1.22e-04]    25.4


#### Вклад отдельных искажений

Практический вопрос: «что будет, если частота уйдёт на 20 кГц?» Отвечаем
цифрой — вероятность ошибки с искажением против вероятности ошибки без него.

**Как читать «во сколько раз хуже»:** для расстройки частоты, фазы и такта
приёмник компенсирует искажение, поэтому отношение близко к 1 — это хороший
результат, компенсатор работает. Для многолучёвости отношение большое, и это
нормально: эквалайзера в тракте нет, а отражения — не шум, мощностью их не
убрать. Такой результат сам по себе является аргументом в обсуждении
требований.

In [9]:
def ber_case(**channel_kw) -> float:
    """BER при текущих параметрах, но с переопределённым каналом."""
    c = cfg.copy()
    c.signal.num_bits = 40_000
    c.channel.seed = 777
    for k, v in channel_kw.items():
        setattr(c.channel, k, v)
    return chain.run(c).ber


base_ber = ber_case()
cases = [
    ("только шум (база)", {}),
    ("CFO 20 кГц", dict(cfo_hz=20_000.0)),
    ("ошибка такта 0.25 симв.", dict(timing_offset_symbols=0.25)),
    ("фаза 35 град", dict(phase_deg=35.0)),
    ("разбаланс I/Q 1.5 дБ", dict(iq_gain_db=1.5, iq_phase_deg=10.0)),
    ("АЦП 6 бит", dict(adc_bits=6, adc_full_scale=1.5)),
    ("многолучёвость 3 луча", dict(multipath=((0.0, 0.0, 0.0), (1.0, -6.0, 45.0), (2.5, -10.0, -90.0)))),
]
print(f"{'случай':28s} {'BER':>11s} {'во сколько раз хуже базы':>26s}")
print("-" * 70)
for label, kw in cases:
    b = ber_case(**kw)
    rel = (b / base_ber) if base_ber > 0 else float("nan")
    print(f"{label:28s} {b:11.3e} {rel:26.2f}")

случай                               BER   во сколько раз хуже базы
----------------------------------------------------------------------


только шум (база)              0.000e+00                        nan


CFO 20 кГц                     2.500e-05                        nan


ошибка такта 0.25 симв.        0.000e+00                        nan


фаза 35 град                   0.000e+00                        nan


разбаланс I/Q 1.5 дБ           1.250e-04                        nan


АЦП 6 бит                      0.000e+00                        nan


многолучёвость 3 луча          4.567e-01                        nan


#### Сравнение схем модуляции

Полезно, когда надо выбрать схему под требования. Напоминание: при равном
Eb/N0 BPSK и QPSK дают **одинаковую** вероятность ошибки; схема с неравными
мощностями квадратур проигрывает из-за перекоса мощности.

In [10]:
bps_map = {"BPSK": 1, "QPSK": 2, "OQPSK": 2, "UQPSK": 2}
eb_ref = 6.0
print(f"Сравнение при Eb/N0 = {eb_ref} дБ, {SWEEP_BITS} бит:")
print(f"{'схема':8s} {'Es/N0':>7s} {'BER модели':>12s} {'BER теории':>12s}")
print("-" * 44)
for scheme in ("BPSK", "QPSK", "OQPSK", "UQPSK"):
    c = cfg.copy()
    c.signal.modulation = scheme
    c.signal.num_bits = SWEEP_BITS
    c.signal.iq_power_ratio_db = 6.0 if scheme == "UQPSK" else 0.0
    c.channel.es_n0_db = eb_ref + 10 * np.log10(bps_map[scheme])
    c.channel.seed = 555
    r = chain.run(c)
    th = float(metrics.ber_theory_from_esn0(scheme, c.channel.es_n0_db, c.signal.iq_power_ratio_db))
    print(f"{scheme:8s} {c.channel.es_n0_db:7.1f} {r.ber:12.2e} {th:12.2e}")

Сравнение при Eb/N0 = 6.0 дБ, 60000 бит:
схема      Es/N0   BER модели   BER теории
--------------------------------------------


BPSK         6.0     2.07e-03     2.39e-03


QPSK         9.0     2.68e-03     2.39e-03


OQPSK        9.0     2.48e-03     2.39e-03


UQPSK        9.0     2.00e-02     1.85e-02


### 6. Квантование и фиксированная точка

Тот же тракт, но с квантованием: АЦП, данные, коэффициенты фильтра — с
заданной разрядностью. Сравниваем с моделью в плавающей точке.

Если результат близок — разрядностей достаточно. Если нет — видно, на каком
этапе теряется точность.

In [11]:
if CHECK_FIXED_POINT:
    res_fixed = chain.run(cfg, fixed=True)
    print(f"{'вариант':24s} {'ошибок':>8s} {'BER':>11s} {'EVM, %':>8s}")
    print("-" * 54)
    print(f"{'float64 (модель)':24s} {res.errors:8d} {res.ber:11.3e} {res.evm_percent:8.1f}")
    print(f"{'fixed-point (ПЛИС)':24s} {res_fixed.errors:8d} {res_fixed.ber:11.3e} {res_fixed.evm_percent:8.1f}")
    print(f"{'теория':24s} {'':>8s} {theory:11.3e}")
    print()
    q_data = fx.QFormat(wl=DATA_WL, fl=DATA_FL)
    print(f"Формат данных Q{DATA_WL}.{DATA_FL}: шаг {q_data.lsb:.2e}, "
          f"диапазон ±{q_data.max_value:.2f}, SQNR {fx.sqnr_theory_db(q_data):.0f} дБ")
    print("Правило: SQNR формата должен быть на 10..15 дБ выше рабочего Es/N0.")
else:
    res_fixed = None
    print("Проверка фиксированной точки выключена (CHECK_FIXED_POINT = False)")

вариант                    ошибок         BER   EVM, %
------------------------------------------------------
float64 (модель)                5   2.500e-05     25.4
fixed-point (ПЛИС)              5   2.500e-05     25.4
теория                              3.430e-05

Формат данных Q16.13: шаг 1.22e-04, диапазон ±4.00, SQNR 98 дБ
Правило: SQNR формата должен быть на 10..15 дБ выше рабочего Es/N0.


### 7. Экспорт данных для реализации

Что выгружается:

- файл коэффициентов фильтра (`.coe`) — для загрузки в память или в IP-ядро;
- тест-векторы (вход и ожидаемый выход) — для сверки реализации с моделью;
- каркас testbench;
- оценка ресурсов и разрядности накопителя.

In [12]:
out_dir = pathlib.Path("out") / "main"

if EXPORT_FPGA:
    q_coeff = fx.QFormat(wl=COEFF_WL, fl=COEFF_FL)
    p_coe = export.save_coe(res.tx.taps, out_dir / "rrc_taps.coe", q_coeff,
                            comment=f"RRC sps={SPS} alpha={ROLLOFF} span={SPAN}")
    print(f"Коэффициенты: {p_coe}")

    n_tb = 4000
    x_tb = sources.complex_noise(n_tb, variance=0.05, seed=3)
    y_tb = filters.fir_filter(np.real(x_tb), res.tx.taps, mode="same")
    files = export.save_test_vectors(out_dir, np.real(x_tb), y_tb, q_coeff, q_coeff,
                                     prefix="fir_rrc", comment="согласованный фильтр RRC")
    for k, v in files.items():
        print(f"Векторы ({k}): {v}")

    tb = export.save_vhdl_testbench(out_dir, "fir_rrc", prefix="fir_rrc",
                                   wl=COEFF_WL, n_samples=n_tb)
    print(f"Каркас testbench: {tb}")
else:
    print("Экспорт выключен (EXPORT_FPGA = False)")

res_info = report.estimate_resources(cfg)
print()
print("ОЦЕНКА РЕСУРСОВ")
for k, v in res_info.items():
    print(f"  {k:34s} {v}")

Коэффициенты: out\main\rrc_taps.coe
Векторы (input): out\main\fir_rrc_in.hex
Векторы (output): out\main\fir_rrc_out.hex
Векторы (info): out\main\fir_rrc_info.txt
Каркас testbench: out\main\fir_rrc_tb.vhd

ОЦЕНКА РЕСУРСОВ
  отводов_RRC                        65
  DSP48_RRC                          33
  LUT_RRC                            390
  FF_RRC                             520
  аккумулятор_бит                    39
  аккумулятор_влезает_в_DSP48        True
  fs_МГц                             4.0
  такт_рекомендуемый_МГц             16.0
  DSP48_всего_оценка                 99


### 8. Техническая сводка

Блок собирает всё, что задано и получено, в один текстовый документ:
параметры сигнала, требования к фильтру, форматы чисел, ресурсы, ожидаемые
характеристики и список проверок.

Его можно распечатать, приложить к письму или использовать как основу
пояснительной записки — там конкретные числа, а не общие слова.

In [13]:
spec_text = report.technical_spec(cfg, res)
print(spec_text)

ТЕХНИЧЕСКАЯ СВОДКА ПО МОДЕЛИ
сформировано автоматически из ExperimentConfig, 2026-11-23 20:15

1. СИГНАЛ
   модуляция ................. QPSK (2 бит/символ)
   битовая скорость .......... 1.0000 Мбит/с
   символьная скорость ....... 0.5000 Мбод
   отсчётов на символ ........ 8
   частота дискретизации ..... 4.0000 МГц
   занимаемая полоса ......... 0.6750 МГц (α = 0.35)
   скважность спектра ........ fs / полоса = 5.93

2. ФОРМИРОВАНИЕ СПЕКТРА
   фильтр .................... RRC, α = 0.35, 8 символов
   отводов ................... 65
   задержка .................. 4.000 симв. = 32.0 отсч.
   коэффициенты .............. Q16.14 (шаг 6.10e-05, SQNR 98 дБ)
   умножителей (симметр.) .... 33 DSP48

3. КАНАЛ, КОТОРЫЙ МОДЕЛИРУЕМ
   Es/N0 ..................... 12.00 дБ (Eb/N0 = 8.99 дБ)
   расстройка несущей (CFO) .. +0.000 кГц
   набег фазы ................ +0.0 град
   фазовый шум ............... 0.0 град СКЗ
   ошибка такта .............. +0.000 симв.
   разбаланс I/Q ............. +0.00 дБ, +

### 9. Сохранение сессии

Сохраняем всё, что нужно, чтобы повторить прогон и подтвердить результат:

- `config.yaml` — параметры (загружаются обратно одной строкой);
- `protocol.txt` — сводка, предупреждения, таблица величин;
- `ber_sweep.csv` — метрики свипа (открывается в табличном редакторе);
- `figures/*.html` — графики (открываются в браузере, печатаются).

In [14]:
if SAVE_SESSION:
    figures = {
        "01_spectrum": plots.fig_spectrum_compare(
            [("передано", wave_tx[:400 * cfg.signal.sps]),
             ("принято", wave_rx[:400 * cfg.signal.sps])],
            fs, "Спектр: переданный и принятый"),
        "02_constellation_after": plots.fig_constellation(
            sym_after, cfg.signal.modulation, "Созвездие после синхронизации",
            iq_power_ratio_db=cfg.signal.iq_power_ratio_db),
        "03_constellation_before": plots.fig_constellation(
            sym_before, cfg.signal.modulation, "Созвездие до синхронизации",
            iq_power_ratio_db=cfg.signal.iq_power_ratio_db),
        "04_filter_response": plots.fig_filter_response(
            res.tx.taps, fs, "АЧХ RRC", symbol_rate=cfg.signal.symbol_rate),
    }
    if mf_out is not None:
        figures["05_eye"] = plots.fig_eye(mf_out, cfg.signal.sps, 2, "Глазковая диаграмма")
    if sweep_rows:
        figures["06_ber"] = plots.fig_ber(
            sweep_rows, cfg.signal.modulation, "Кривая BER",
            iq_power_ratio_db=cfg.signal.iq_power_ratio_db)
    if phase_hist is not None:
        figures["07_costas"] = plots.fig_phase_track(
            {"история фазы": res.rx.stages["история_фазы"],
             "история частоты": res.rx.stages["история_частоты"]},
            "Работа петли Костаса")

    saved = report.save_session(out_dir, cfg, res, sweep_rows=sweep_rows,
                                figures=figures, spec_text=spec_text)
else:
    print("Сохранение выключено (SAVE_SESSION = False)")

Сессия сохранена в: out\main
   config         -> out\main\config.yaml
   protocol       -> out\main\protocol.txt
   csv            -> out\main\ber_sweep.csv
   fig_01_spectrum -> out\main\figures\01_spectrum.html
   fig_02_constellation_after -> out\main\figures\02_constellation_after.html
   fig_03_constellation_before -> out\main\figures\03_constellation_before.html
   fig_04_filter_response -> out\main\figures\04_filter_response.html
   fig_05_eye     -> out\main\figures\05_eye.html
   fig_06_ber     -> out\main\figures\06_ber.html
   fig_07_costas  -> out\main\figures\07_costas.html


### 10. Настройка под другую задачу

Шпаргалка: какой параметр за что отвечает и куда смотреть, если результат
не тот.

| Что нужно | Что менять | На что смотреть |
|---|---|---|
| другая скорость | `BIT_RATE` | полоса, `fs`, ресурсы |
| другая модуляция | `MODULATION` | BER, пик-фактор, ресурсы |
| уже полоса | `ROLLOFF` (меньше) | длина фильтра, чувствительность к такту |
| хуже канал | `ES_N0_DB` (меньше) | BER, форма кривой |
| есть расстройка частоты | `CFO_HZ` | оценка CFO, полоса петли |
| есть отражения | `MULTIPATH` | BER, созвездие |
| реальный АЦП | `ADC_BITS`, `ADC_FULL_SCALE` | ограничение шкалы, BER |
| аппаратная реализация | `DATA_WL`, `COEFF_WL` | деградация от квантования, ресурсы |
| не сходится BER | `LOOP_BW_CARRIER`, `SPS` | траектория петли, глаз |

#### Что уточнить перед моделированием реальной системы

1. Точная схема модуляции и скорости (символьная, битовая); есть ли кодирование.
2. Формат кадра: преамбула или маркер, длина, служебные поля.
3. Требования: какая вероятность ошибки при каком Eb/N0, какие допустимые потери.
4. Аппаратная платформа: целевой такт, доступные умножители и память.
5. АЦП: разрядность, частота, интерфейс, полная шкала.
6. Что уже есть: существующая модель, документация, что можно взять за основу.

#### Если что-то не сходится — порядок разбора

1. Выключить шум: ошибок должно быть 0 (проверка модуляции и фильтрации).
2. Включить только шум: результат должен совпасть с теорией (проверка
   калибровки отношения сигнал/шум).
3. Включать искажения по одному и смотреть, после какого ломается.
4. Проверить квантование — не портит ли оно результат.
5. Смотреть созвездие и глазковую диаграмму: они говорят правду быстрее цифр.